# NB1 — Collect raw data (`01-collect-data`)
**Project:** *Is AI Changing Who Gets Hired?* — see `RESEARCH_PLAN.md` and `KAGGLE_IMPLEMENTATION.md`.

**Before you run:** right panel → **Settings → Internet: On**, **Accelerator: None**. Phone verification is needed for Internet.

**What it downloads (all into `/kaggle/working/`):**

| Cell | Source | Output files |
|---|---|---|
| 2 | Hacker News "Who is hiring?" (Algolia API) | `hn_threads.csv`, `hn_jobs.csv` |
| 3 | FRED (Indeed US index, JOLTS, Fed funds rate) | `fred_series.csv` |
| 4 | Indeed Hiring Lab GitHub (10 countries + euro area) | `indeed_aggregate.csv`, `indeed_sector.csv`, `indeed_sector_job_title_examples.csv` |
| 5 | O\*NET (latest text release) | `onet_*.csv` (every table in the release) |
| 6 | AI exposure: GPTs-are-GPTs + Felten AIOE | `exposure_eloundou.csv`, `aioe/…` |
| 7 | Anthropic Economic Index (latest release) | `anthropic/…` |
| 8 | *(optional, off by default)* USAJOBS historic job notices | `usajobs/usajobs_YYYY-MM.csv` |
| 9 | Record of what was downloaded | `SOURCES.md`, `manifest.csv` |

**Run time:** about 10–20 minutes without USAJOBS (most of it is HN and the Anthropic files, ~300 MB).

**When done:** **Save Version → Save & Run All (Commit)** → open the version's **Output** tab → **New Dataset** → name it **`jobai-raw`**.

In [ ]:
# Cell 1 — setup
import os, re, io, time, json, html, zipfile
import requests, pandas as pd, numpy as np

OUT = "/kaggle/working"
START = "2019-01-01"          # first month of the study window
TODAY = pd.Timestamp.today().normalize()
CURRENT_MONTH = TODAY.strftime("%Y-%m")   # this month is still incomplete -> flagged, dropped in NB2

S = requests.Session()
S.headers.update({"User-Agent": "academic-research-notebook (Kaggle)"})

def get(url, params=None, timeout=120, tries=5, stream=False):
    """GET with retries and backoff for rate limits / server errors."""
    for i in range(tries):
        try:
            r = S.get(url, params=params, timeout=timeout, stream=stream)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except Exception as e:
            if i == tries - 1:
                raise
            wait = 2 ** i * 3
            print(f"  retry {i+1}/{tries-1} in {wait}s ({e}) — {url}")
            time.sleep(wait)

LOG = {}   # facts for SOURCES.md
print("Download date:", TODAY.date(), "| incomplete current month:", CURRENT_MONTH)

## Cell 2 — Hacker News "Who is hiring?" job ads (main text data)
One thread per month; every **top-level** comment is one job ad. Monthly hiring threads only ("Who wants to be hired?" and "Freelancer" threads are excluded).
`complete_month = False` marks the current month, whose thread is still filling up — NB2 drops it.

In [ ]:
# Cell 2 — HN threads, then every top-level comment
BASE = "https://hn.algolia.com/api/v1"
start_ts = int(pd.Timestamp(START).timestamp())

r = get(f"{BASE}/search_by_date",
        params={"tags": "story,author_whoishiring", "hitsPerPage": 1000,
                "numericFilters": f"created_at_i>={start_ts}"}).json()
threads = pd.DataFrame([{"thread_id": h["objectID"], "title": h["title"], "created_at": h["created_at"]}
                        for h in r["hits"]])
threads = threads[threads["title"].str.contains(r"^Ask HN: Who is hiring\?", case=False, regex=True)].copy()
threads["created_at"] = pd.to_datetime(threads["created_at"], utc=True)
threads["month"] = threads["created_at"].dt.strftime("%Y-%m")
threads = (threads.sort_values("created_at")
                  .drop_duplicates("month", keep="first")       # one thread per month
                  .reset_index(drop=True))
threads["complete_month"] = threads["month"] < CURRENT_MONTH
threads.to_csv(f"{OUT}/hn_threads.csv", index=False)
print(len(threads), "threads:", threads["month"].min(), "→", threads["month"].max())

expected = pd.period_range(START, CURRENT_MONTH, freq="M").strftime("%Y-%m")
missing = sorted(set(expected) - set(threads["month"]))
print("Missing months:", missing or "none")

def clean(t):
    t = html.unescape(t or "")
    t = re.sub(r"<p>", "\n", t)
    t = re.sub(r"<[^>]+>", " ", t)
    return re.sub(r"[ \t]+", " ", t).strip()

rows = []
for i, t in enumerate(threads.itertuples(), 1):
    item = get(f"{BASE}/items/{t.thread_id}", timeout=180).json()
    for c in item.get("children") or []:
        if c.get("text"):                                    # skip deleted / dead comments
            rows.append({"month": t.month, "thread_id": t.thread_id, "comment_id": c["id"],
                         "author": c.get("author"), "posted_at": c.get("created_at"),
                         "complete_month": t.complete_month, "text": clean(c["text"])})
    if i % 10 == 0 or i == len(threads):
        print(f"  {i}/{len(threads)} threads, {len(rows):,} ads so far")
    time.sleep(1)                                            # be polite to the API

hn = pd.DataFrame(rows).drop_duplicates("comment_id")
hn.to_csv(f"{OUT}/hn_jobs.csv", index=False)
per_month = hn.groupby("month").size()
print(hn.shape, "| ads per month: median", int(per_month.median()), "min", int(per_month.min()))
low = per_month[per_month < 200]
if len(low):
    print("⚠ months with < 200 ads (check these):\n", low)
LOG["hn"] = f"{len(threads)} monthly threads {threads['month'].min()}–{threads['month'].max()}, {len(hn):,} top-level ads"
per_month.tail(12)

## Cell 3 — FRED time series (whole US market)
`FEDFUNDS` is included because the 2022–23 interest-rate rises are the main confounder for the AI effect.
Indeed data starts **1 Feb 2020 = 100** (there is no 2019).

In [ ]:
# Cell 3 — FRED
SERIES = {
    "IHLIDXUS": "Indeed job postings index, US total (Feb 2020 = 100)",
    "JTSJOL":   "JOLTS job openings, total nonfarm (thousands)",
    "JTSHIL":   "JOLTS hires, total nonfarm (thousands)",
    "FEDFUNDS": "Effective federal funds rate (%) — interest-rate confounder",
}
def fred(series_id):
    # FRED hangs on custom/browser User-Agents but answers the default python-requests one,
    # so use a plain requests.get here (not the shared session).
    for i in range(4):
        try:
            r = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                             params={"id": series_id}, timeout=60)
            r.raise_for_status()
            break
        except Exception as e:
            if i == 3:
                raise
            print(f"  FRED retry {i+1}/3 ({e})"); time.sleep(5 * (i + 1))
    df = pd.read_csv(io.StringIO(r.text))
    df.columns = ["date", series_id]                     # first column is the date, whatever its name
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")   # "." = missing
    return df

# Backup for JOLTS if FRED fails: BLS public API v1 (no key; 10 years per request)
BLS_IDS = {"JTSJOL": "JTS000000000000000JOL", "JTSHIL": "JTS000000000000000HIL"}
def bls(series_id):
    rows = []
    for y0 in (2018, 2028):
        body = {"seriesid": [BLS_IDS[series_id]], "startyear": str(y0), "endyear": str(min(y0 + 9, TODAY.year))}
        if y0 > TODAY.year:
            break
        d = S.post("https://api.bls.gov/publicAPI/v1/timeseries/data/", json=body, timeout=60).json()
        for p in d["Results"]["series"][0]["data"]:
            if p["period"].startswith("M") and p["period"] != "M13":
                rows.append({"date": pd.Timestamp(int(p["year"]), int(p["period"][1:]), 1),
                             series_id: pd.to_numeric(p["value"], errors="coerce")})
    return pd.DataFrame(rows)

fred_df, failed = None, []
for sid in SERIES:
    try:
        d = fred(sid); src = "FRED"
    except Exception as e:
        if sid in BLS_IDS:
            try:
                d = bls(sid); src = "BLS API (FRED failed)"
            except Exception as e2:
                failed.append(sid); print(f"✗ {sid}: FRED and BLS both failed ({e2})"); continue
        else:
            failed.append(sid); print(f"✗ {sid}: FRED failed ({e}) — continuing without it"); continue
    print(f"{sid:9s} {d['date'].min().date()} → {d['date'].max().date()}  [{src}]  ({SERIES[sid]})")
    fred_df = d if fred_df is None else fred_df.merge(d, on="date", how="outer")
if failed:
    print("⚠ Missing series:", failed, "— re-run this cell later; IHLIDXUS is also in indeed_aggregate.csv (US)")
fred_df = fred_df[fred_df["date"] >= "2018-01-01"].sort_values("date")
fred_df.to_csv(f"{OUT}/fred_series.csv", index=False)
LOG["fred"] = ", ".join(SERIES)
fred_df.tail()

## Cell 4 — Indeed Hiring Lab (countries and sectors)
Daily index, Feb 2020 = 100. **Sector** data exists only for US, GB, CA, DE, FR, AU; IE, ES, IT, NL and the euro area (EA) have the total index only.
India is **not** covered by Indeed Hiring Lab, so RQ4 is limited to these countries.
`sector-job-title-examples.csv` lists example job titles per Indeed sector — use it to build the sector ↔ SOC crosswalk.

In [ ]:
# Cell 4 — Indeed Hiring Lab GitHub
RAW = "https://raw.githubusercontent.com/hiring-lab/job_postings_tracker/master"
AGG_COUNTRIES    = ["US", "GB", "CA", "DE", "FR", "AU", "IE", "ES", "IT", "NL", "EA"]
SECTOR_COUNTRIES = ["US", "GB", "CA", "DE", "FR", "AU"]

def fetch_many(paths):
    frames = []
    for p in paths:
        try:
            frames.append(pd.read_csv(io.StringIO(get(f"{RAW}/{p}").text)))
        except Exception as e:
            print("  ✗", p, e)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

agg = fetch_many([f"{c}/aggregate_job_postings_{c}.csv" for c in AGG_COUNTRIES])
sec = fetch_many([f"{c}/job_postings_by_sector_{c}.csv" for c in SECTOR_COUNTRIES])
agg.to_csv(f"{OUT}/indeed_aggregate.csv", index=False)
sec.to_csv(f"{OUT}/indeed_sector.csv", index=False)
pd.read_csv(io.StringIO(get(f"{RAW}/sector-job-title-examples.csv").text)) \
  .to_csv(f"{OUT}/indeed_sector_job_title_examples.csv", index=False)

print("aggregate:", agg.shape, "| countries:", sorted(agg["jobcountry"].unique()) if "jobcountry" in agg else "?")
print("sector:   ", sec.shape, "| sectors:", sec["display_name"].nunique() if "display_name" in sec else "?")
print("date range:", sec["date"].min(), "→", sec["date"].max())
LOG["indeed"] = f"aggregate {AGG_COUNTRIES}; sector {SECTOR_COUNTRIES}; to {sec['date'].max()}"
sec.head()

## Cell 5 — O\*NET (latest release)
Finds the newest version on onetcenter.org automatically (31.0 as of Nov 2026). Set `ONET_VERSION = "31.0"` to pin it.
Every table in the release is saved as `onet_<Table_Name>.csv` (spaces → `_`), so NB2/NB4 can load
`onet_Occupation_Data.csv`, `onet_Alternate_Titles.csv`, `onet_Technology_Skills.csv`, `onet_Skills.csv`, `onet_Task_Statements.csv`.
The O\*NET server can be slow — the download allows up to 10 minutes and checks the file arrived complete.

In [ ]:
# Cell 5 — O*NET
ONET_VERSION = None      # e.g. "31.0" to pin; None = newest listed on the website

if ONET_VERSION is None:
    page = get("https://www.onetcenter.org/database.html").text
    versions = re.findall(r"/dictionary/(\d+)\.(\d+)/", page) + re.findall(r"db_(\d+)_(\d+)_", page)
    ONET_VERSION = ".".join(map(str, max((int(a), int(b)) for a, b in versions)))
url = f"https://www.onetcenter.org/dl_files/database/db_{ONET_VERSION.replace('.', '_')}_text.zip"
print("O*NET", ONET_VERSION, "→", url)

r = get(url, timeout=600, stream=True)
expected = int(r.headers.get("Content-Length", 0))
buf = io.BytesIO()
for chunk in r.iter_content(1 << 20):
    buf.write(chunk)
assert not expected or buf.tell() == expected, f"incomplete download: {buf.tell():,} of {expected:,} bytes — re-run this cell"
z = zipfile.ZipFile(buf)

saved = []
for name in z.namelist():
    if not name.endswith(".txt"):
        continue
    table = os.path.basename(name)[:-4]
    try:
        df = pd.read_csv(z.open(name), sep="\t", dtype=str, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(z.open(name), sep="\t", dtype=str, encoding="latin-1")
    fn = f"onet_{table.replace(' ', '_').replace(',', '')}.csv"
    df.to_csv(f"{OUT}/{fn}", index=False)
    saved.append((fn, len(df)))
print(len(saved), "tables saved")

NEEDED = ["onet_Occupation_Data.csv", "onet_Alternate_Titles.csv", "onet_Technology_Skills.csv",
          "onet_Skills.csv", "onet_Task_Statements.csv"]
have = {f for f, _ in saved}
for f in NEEDED:
    print(("✓ " if f in have else "✗ MISSING — check the table list below and update NB2/NB4: ") + f)
LOG["onet"] = f"O*NET {ONET_VERSION} text release, {len(saved)} tables"
print(pd.DataFrame(saved, columns=["file", "rows"]).sort_values("file").to_string(index=False))

## Cell 6 — AI exposure scores
* **Eloundou et al. "GPTs are GPTs"** — occupation level, keyed on O\*NET-SOC code. Columns `dv_rating_*` = GPT-4-rated, `human_rating_*` = human-rated; α/β/γ = exposure definitions (β is the usual headline measure).
* **Felten, Raj & Seamans AIOE** — the authors' data appendix (Excel). Each sheet is also saved as CSV.

In [ ]:
# Cell 6 — exposure
el = pd.read_csv(io.StringIO(get("https://raw.githubusercontent.com/openai/GPTs-are-GPTs/main/data/occ_level.csv").text))
el.to_csv(f"{OUT}/exposure_eloundou.csv", index=False)
print("Eloundou:", el.shape, list(el.columns))

os.makedirs(f"{OUT}/aioe", exist_ok=True)
x = get("https://raw.githubusercontent.com/AIOE-Data/AIOE/main/AIOE_DataAppendix.xlsx").content
with open(f"{OUT}/aioe/AIOE_DataAppendix.xlsx", "wb") as fh:
    fh.write(x)
for sheet, df in pd.read_excel(io.BytesIO(x), sheet_name=None).items():
    fn = re.sub(r"[^A-Za-z0-9]+", "_", sheet).strip("_")
    df.to_csv(f"{OUT}/aioe/aioe_{fn}.csv", index=False)
    print(f"AIOE sheet '{sheet}':", df.shape)
LOG["exposure"] = "GPTs-are-GPTs occ_level.csv; Felten AIOE_DataAppendix.xlsx"
el.head()

## Cell 7 — Anthropic Economic Index (latest release)
Downloads the newest `release_YYYY_MM_DD` folder from Hugging Face (about 300 MB for the 2026-06-26 release).
Read its `data_documentation.md` before using the columns. Set `AEI_RELEASE = "release_2026_06_26"` to pin a release.

In [ ]:
# Cell 7 — Anthropic Economic Index
from huggingface_hub import list_repo_files, hf_hub_download
REPO = "Anthropic/EconomicIndex"
AEI_RELEASE = None       # None = newest

files = list_repo_files(REPO, repo_type="dataset")
releases = sorted({f.split("/")[0] for f in files if f.startswith("release_")})
print("Releases available:", releases)
AEI_RELEASE = AEI_RELEASE or releases[-1]

to_get = [f for f in files if f.startswith(AEI_RELEASE + "/")
          and f.endswith((".csv", ".md", ".json", ".parquet"))]
for f in to_get:
    p = hf_hub_download(REPO, f, repo_type="dataset", local_dir=f"{OUT}/anthropic")
    print(f"  {f}  ({os.path.getsize(p)/1e6:.1f} MB)")
LOG["anthropic"] = f"{REPO} {AEI_RELEASE} ({len(to_get)} files)"

## Cell 8 — *(optional)* USAJOBS historic job notices
US federal job notices of **all** job types — the backup control group if HN has too few low-exposure jobs (see NB2 step 5).
No API key needed. Fields: title, open date, **OPM job series** (map to SOC with an OPM ↔ SOC crosswalk in NB2), grade, agency.
⚠ It has **no ad text**, so it helps RQ1 (counts) but not RQ2 (skills).

**Size:** about 35–40k notices per month → ~3.5 million rows and ~7,000 API calls for 2019–2026, i.e. several hours.
Run it a few years at a time (`USAJOBS_YEARS`). Each month is saved to its own file and skipped on re-runs —
but `/kaggle/working` is wiped between sessions, so commit (**Save Version**) after each batch.

In [ ]:
# Cell 8 — USAJOBS historic JOA (off by default)
RUN_USAJOBS = False
USAJOBS_YEARS = [2022, 2023]          # run in batches, e.g. [2019, 2020] then [2021, 2022] ...

KEEP = ["usajobsControlNumber", "positionTitle", "positionOpenDate", "positionCloseDate",
        "minimumGrade", "maximumGrade", "payScale", "hiringDepartmentName", "hiringAgencyName",
        "appointmentType", "workSchedule", "teleworkEligible", "totalOpenings", "whoMayApply"]

def usajobs_month(y, m):
    first = pd.Timestamp(y, m, 1); last = first + pd.offsets.MonthEnd(0)
    url = "https://data.usajobs.gov/api/historicjoa"
    params = {"StartPositionOpenDate": first.strftime("%m-%d-%Y"),
              "EndPositionOpenDate":   last.strftime("%m-%d-%Y")}
    rows = []
    while True:
        d = get(url, params=params, timeout=180).json()
        for j in d.get("data", []):
            row = {k: j.get(k) for k in KEEP}
            row["series"] = ";".join(c.get("series", "") for c in (j.get("jobcategories") or []))
            rows.append(row)
        nxt = (d.get("paging") or {}).get("next")
        if not nxt:
            return pd.DataFrame(rows, columns=KEEP + ["series"])
        url, params = "https://data.usajobs.gov" + nxt, None
        time.sleep(0.5)

if RUN_USAJOBS:
    os.makedirs(f"{OUT}/usajobs", exist_ok=True)
    for y in USAJOBS_YEARS:
        for m in range(1, 13):
            ym = f"{y}-{m:02d}"
            fn = f"{OUT}/usajobs/usajobs_{ym}.csv"
            if ym >= CURRENT_MONTH or os.path.exists(fn):
                continue
            t0 = time.time()
            df = usajobs_month(y, m)
            df = df[~df["positionTitle"].fillna("").str.contains(r"\bTEST\b")]   # drop template/test notices
            df.to_csv(fn, index=False)
            print(f"  {ym}: {len(df):,} notices ({time.time()-t0:.0f}s)")
    LOG["usajobs"] = f"historic JOA, years {USAJOBS_YEARS}"
else:
    print("USAJOBS skipped (set RUN_USAJOBS = True to run).")

## Cell 9 — record sources and a file manifest
Reviewers will ask exactly what was downloaded and when. `SOURCES.md` is written from what actually ran.

In [ ]:
# Cell 9 — SOURCES.md + manifest
manifest = []
for root, _, fs in os.walk(OUT):
    if "/.cache" in root or "/.ipynb" in root:
        continue
    for f in fs:
        p = os.path.join(root, f)
        manifest.append({"file": os.path.relpath(p, OUT), "MB": round(os.path.getsize(p) / 1e6, 2)})
manifest = pd.DataFrame(manifest).sort_values("file")
manifest.to_csv(f"{OUT}/manifest.csv", index=False)

refs = {
    "hn":        "Hacker News 'Ask HN: Who is hiring?' via HN Algolia API (hn.algolia.com/api)",
    "fred":      "FRED, Federal Reserve Bank of St. Louis (fred.stlouisfed.org)",
    "indeed":    "Indeed Hiring Lab, github.com/hiring-lab/job_postings_tracker (Feb 2020 = 100)",
    "onet":      "O*NET database, onetcenter.org (CC BY 4.0)",
    "exposure":  "Eloundou et al. github.com/openai/GPTs-are-GPTs; Felten, Raj & Seamans github.com/AIOE-Data/AIOE",
    "anthropic": "Anthropic Economic Index, huggingface.co/datasets/Anthropic/EconomicIndex",
    "usajobs":   "USAJOBS Historic JOA API, data.usajobs.gov/api/historicjoa",
}
with open(f"{OUT}/SOURCES.md", "w") as fh:
    fh.write(f"# Raw data sources\n\nDownloaded on {TODAY.date()} (Kaggle notebook 01-collect-data).\n")
    fh.write(f"Current month {CURRENT_MONTH} is incomplete and flagged `complete_month = False` in HN files.\n\n")
    for k, src in refs.items():
        if k in LOG:
            fh.write(f"- **{src}** — {LOG[k]}\n")
print(open(f"{OUT}/SOURCES.md").read())
print(f"Total: {manifest['MB'].sum():.0f} MB in {len(manifest)} files")
print(manifest.to_string(index=False))